In [ ]:
# ============================================================
# STAGE 2A:
# I-JEPA PRETRAINED ENCODER + CLASSIFICATION HEAD
# ============================================================
## import os in the code 
import os
## importing json 
import json
## importing math
import math
## importing random
import random
## importing shutil 
import shutil
import warnings
from copy import deepcopy
from pathlib import Path
from dataclasses import dataclass, asdict
from typing import Optional, Dict, List, Tuple, Any

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from PIL import Image, ImageFile
from tqdm.auto import tqdm

import torch
import torch.nn as nn
import torch.nn.functional as F

from torch.utils.data import (
    Dataset,
    DataLoader,
    Subset,
    WeightedRandomSampler
)

from torchvision import datasets, transforms

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import label_binarize
from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix,
    classification_report,
    roc_curve,
    auc,
    roc_auc_score,
    precision_recall_curve,
    average_precision_score
)


ImageFile.LOAD_TRUNCATED_IMAGES = True
warnings.filterwarnings("ignore")


# ============================================================
# 1. CONFIGURATION
# ============================================================

@dataclass
class ClassificationConfig:

    # --------------------------------------------------------
    # Paths
    # --------------------------------------------------------
    DATA_DIR: str = "Splited/Fetal_planes_splited/Split_40"

    PRETRAINED_ENCODER_PATH: str = (
        "checkpoints_ultrasound_ijepa/"
        "final_ijepa_encoder.pth"
    )

    SAVE_DIR: str = "stage2a_classification_results"

    # --------------------------------------------------------
    # Data splitting
    # --------------------------------------------------------
    # True:
    # DATA_DIR contains class folders only.
    #
    # False:
    # DATA_DIR contains train, val, and test folders.
    AUTO_SPLIT: bool = True

    TRAIN_RATIO: float = 0.70
    VAL_RATIO: float = 0.15
    TEST_RATIO: float = 0.15

    # --------------------------------------------------------
    # Encoder architecture
    # Must match Stage 1 exactly
    # --------------------------------------------------------
    IMG_SIZE: int = 224
    PATCH_SIZE: int = 16
    IN_CHANNELS: int = 3

    EMBED_DIM: int = 384
    ENCODER_DEPTH: int = 8
    ENCODER_HEADS: int = 6
    MLP_RATIO: float = 4.0
    DROPOUT: float = 0.0

    # --------------------------------------------------------
    # Classification head
    # --------------------------------------------------------
    CLASSIFIER_HIDDEN_DIM: int = 256
    CLASSIFIER_DROPOUT: float = 0.35

    # --------------------------------------------------------
    # Training
    # --------------------------------------------------------
    BATCH_SIZE: int = 32
    NUM_WORKERS: int = 4

    # Phase 1: train classifier only
    HEAD_WARMUP_EPOCHS: int = 5
    HEAD_LR: float = 3e-4

    # Phase 2: unfreeze final transformer blocks
    PARTIAL_FINETUNE_EPOCHS: int = 15
    NUM_UNFROZEN_BLOCKS: int = 2

    PARTIAL_ENCODER_LR: float = 1e-5
    PARTIAL_HEAD_LR: float = 1e-4

    # Phase 3: full fine-tuning
    FULL_FINETUNE_EPOCHS: int = 30

    FULL_ENCODER_LR: float = 3e-6
    FULL_HEAD_LR: float = 5e-5

    WEIGHT_DECAY: float = 1e-4
    LABEL_SMOOTHING: float = 0.05

    GRADIENT_CLIP_NORM: float = 1.0

    EARLY_STOPPING_PATIENCE: int = 10

    # Class imbalance
    USE_CLASS_WEIGHTS: bool = True
    USE_WEIGHTED_SAMPLER: bool = False

    # --------------------------------------------------------
    # Runtime
    # --------------------------------------------------------
    USE_AMP: bool = True
    PIN_MEMORY: bool = True
    PERSISTENT_WORKERS: bool = True
    PREFETCH_FACTOR: int = 2

    SEED: int = 42


CFG = ClassificationConfig()

CFG.GRID_SIZE = CFG.IMG_SIZE // CFG.PATCH_SIZE
CFG.NUM_PATCHES = CFG.GRID_SIZE ** 2

DEVICE = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

os.makedirs(CFG.SAVE_DIR, exist_ok=True)


# ============================================================
# 2. REPRODUCIBILITY
# ============================================================

def set_seed(seed: int) -> None:
    random.seed(seed)
    np.random.seed(seed)

    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

    torch.backends.cudnn.benchmark = True
    torch.backends.cudnn.deterministic = False


set_seed(CFG.SEED)


# ============================================================
# 3. FIXED 2D SINE-COSINE POSITIONAL EMBEDDING
# ============================================================

def get_1d_sincos_embedding(
    embed_dim: int,
    positions: np.ndarray
) -> np.ndarray:

    if embed_dim % 2 != 0:
        raise ValueError(
            "The 1D positional embedding dimension must be even."
        )

    omega = np.arange(
        embed_dim // 2,
        dtype=np.float64
    )

    omega = omega / (embed_dim / 2.0)
    omega = 1.0 / (10000 ** omega)

    positions = positions.reshape(-1)

    output = np.einsum(
        "m,d->md",
        positions,
        omega
    )

    embedding_sin = np.sin(output)
    embedding_cos = np.cos(output)

    return np.concatenate(
        [embedding_sin, embedding_cos],
        axis=1
    )


def get_2d_sincos_embedding(
    embed_dim: int,
    grid_size: int
) -> torch.Tensor:

    if embed_dim % 4 != 0:
        raise ValueError(
            "EMBED_DIM must be divisible by 4."
        )

    grid_h = np.arange(
        grid_size,
        dtype=np.float32
    )

    grid_w = np.arange(
        grid_size,
        dtype=np.float32
    )

    grid = np.meshgrid(
        grid_w,
        grid_h
    )

    grid_w_flat = grid[0].reshape(-1)
    grid_h_flat = grid[1].reshape(-1)

    embedding_h = get_1d_sincos_embedding(
        embed_dim // 2,
        grid_h_flat
    )

    embedding_w = get_1d_sincos_embedding(
        embed_dim // 2,
        grid_w_flat
    )

    embedding = np.concatenate(
        [embedding_h, embedding_w],
        axis=1
    )

    return torch.tensor(
        embedding,
        dtype=torch.float32
    ).unsqueeze(0)


# ============================================================
# 4. I-JEPA ViT ENCODER
# Must match Stage 1
# ============================================================

class PatchEmbedding(nn.Module):

    def __init__(self):
        super().__init__()

        self.projection = nn.Conv2d(
            in_channels=CFG.IN_CHANNELS,
            out_channels=CFG.EMBED_DIM,
            kernel_size=CFG.PATCH_SIZE,
            stride=CFG.PATCH_SIZE
        )

    def forward(
        self,
        images: torch.Tensor
    ) -> torch.Tensor:

        tokens = self.projection(images)

        tokens = tokens.flatten(2).transpose(1, 2)

        return tokens


class FeedForwardNetwork(nn.Module):

    def __init__(
        self,
        embedding_dim: int,
        mlp_ratio: float,
        dropout: float
    ):
        super().__init__()

        hidden_dim = int(
            embedding_dim * mlp_ratio
        )

        self.network = nn.Sequential(
            nn.Linear(
                embedding_dim,
                hidden_dim
            ),
            nn.GELU(),
            nn.Dropout(dropout),

            nn.Linear(
                hidden_dim,
                embedding_dim
            ),
            nn.Dropout(dropout)
        )

    def forward(
        self,
        tokens: torch.Tensor
    ) -> torch.Tensor:

        return self.network(tokens)


class TransformerBlock(nn.Module):

    def __init__(
        self,
        embedding_dim: int,
        num_heads: int,
        mlp_ratio: float,
        dropout: float
    ):
        super().__init__()

        self.norm1 = nn.LayerNorm(
            embedding_dim
        )

        self.attention = nn.MultiheadAttention(
            embed_dim=embedding_dim,
            num_heads=num_heads,
            dropout=dropout,
            batch_first=True
        )

        self.norm2 = nn.LayerNorm(
            embedding_dim
        )

        self.mlp = FeedForwardNetwork(
            embedding_dim=embedding_dim,
            mlp_ratio=mlp_ratio,
            dropout=dropout
        )

    def forward(
        self,
        tokens: torch.Tensor
    ) -> torch.Tensor:

        normalized_tokens = self.norm1(tokens)

        attention_output, _ = self.attention(
            normalized_tokens,
            normalized_tokens,
            normalized_tokens,
            need_weights=False
        )

        tokens = tokens + attention_output

        tokens = tokens + self.mlp(
            self.norm2(tokens)
        )

        return tokens


class ViTEncoder(nn.Module):

    def __init__(self):
        super().__init__()

        self.patch_embedding = PatchEmbedding()

        position_embedding = get_2d_sincos_embedding(
            embed_dim=CFG.EMBED_DIM,
            grid_size=CFG.GRID_SIZE
        )

        self.register_buffer(
            "position_embedding",
            position_embedding,
            persistent=True
        )

        self.blocks = nn.ModuleList([
            TransformerBlock(
                embedding_dim=CFG.EMBED_DIM,
                num_heads=CFG.ENCODER_HEADS,
                mlp_ratio=CFG.MLP_RATIO,
                dropout=CFG.DROPOUT
            )
            for _ in range(CFG.ENCODER_DEPTH)
        ])

        self.norm = nn.LayerNorm(
            CFG.EMBED_DIM
        )

    def forward(
        self,
        images: torch.Tensor
    ) -> torch.Tensor:

        tokens = self.patch_embedding(images)

        tokens = tokens + self.position_embedding.to(
            device=tokens.device,
            dtype=tokens.dtype
        )

        for block in self.blocks:
            tokens = block(tokens)

        tokens = self.norm(tokens)

        return tokens


# ============================================================
# 5. ROBUST I-JEPA CHECKPOINT LOADING
# ============================================================

def remove_prefix(
    state_dict: Dict[str, torch.Tensor],
    prefix: str
) -> Dict[str, torch.Tensor]:

    return {
        key[len(prefix):] if key.startswith(prefix) else key: value
        for key, value in state_dict.items()
    }


def extract_encoder_state_dict(
    checkpoint: Any
) -> Dict[str, torch.Tensor]:

    if not isinstance(checkpoint, dict):
        raise TypeError(
            "Checkpoint must be a dictionary."
        )

    if "encoder_state_dict" in checkpoint:
        state_dict = checkpoint["encoder_state_dict"]

    elif "student_encoder" in checkpoint:
        state_dict = checkpoint["student_encoder"]

    elif "teacher_encoder" in checkpoint:
        state_dict = checkpoint["teacher_encoder"]

    elif "model_state_dict" in checkpoint:

        complete_state = checkpoint["model_state_dict"]

        state_dict = {
            key.replace(
                "encoder.",
                "",
                1
            ): value
            for key, value in complete_state.items()
            if key.startswith("encoder.")
        }

        if not state_dict:
            state_dict = complete_state

    elif all(
        torch.is_tensor(value)
        for value in checkpoint.values()
    ):
        state_dict = checkpoint

    else:
        raise KeyError(
            "No compatible encoder state dictionary was found."
        )

    possible_prefixes = [
        "module.",
        "student_encoder.",
        "teacher_encoder.",
        "encoder.",
        "backbone.encoder.",
        "backbone."
    ]

    for prefix in possible_prefixes:
        if any(
            key.startswith(prefix)
            for key in state_dict
        ):
            state_dict = remove_prefix(
                state_dict,
                prefix
            )

    return state_dict


def load_pretrained_ijepa_encoder(
    encoder: ViTEncoder,
    checkpoint_path: str,
    strict: bool = True
) -> ViTEncoder:

    checkpoint_path = Path(
        checkpoint_path
    ).expanduser().resolve()

    if not checkpoint_path.exists():
        raise FileNotFoundError(
            f"Encoder checkpoint was not found:\n"
            f"{checkpoint_path}"
        )

    print(
        f"Loading pretrained encoder:\n"
        f"{checkpoint_path}"
    )

    checkpoint = torch.load(
        checkpoint_path,
        map_location="cpu",
        weights_only=False
    )

    state_dict = extract_encoder_state_dict(
        checkpoint
    )

    result = encoder.load_state_dict(
        state_dict,
        strict=strict
    )

    print("Pretrained I-JEPA encoder loaded.")

    if not strict:
        print(
            "Missing keys:",
            result.missing_keys
        )

        print(
            "Unexpected keys:",
            result.unexpected_keys
        )

    return encoder


# ============================================================
# 6. CLASSIFICATION MODEL
# ============================================================

class IJEPAAttentionPool(nn.Module):
    """
    Learns a weighted global representation from patch tokens.
    """

    def __init__(
        self,
        embed_dim: int
    ):
        super().__init__()

        self.attention = nn.Sequential(
            nn.Linear(embed_dim, embed_dim // 2),
            nn.Tanh(),
            nn.Linear(embed_dim // 2, 1)
        )

    def forward(
        self,
        tokens: torch.Tensor
    ) -> Tuple[torch.Tensor, torch.Tensor]:

        attention_logits = self.attention(
            tokens
        ).squeeze(-1)

        attention_weights = torch.softmax(
            attention_logits,
            dim=1
        )

        pooled = torch.sum(
            tokens * attention_weights.unsqueeze(-1),
            dim=1
        )

        return pooled, attention_weights


class IJEPAClassifier(nn.Module):

    def __init__(
        self,
        num_classes: int
    ):
        super().__init__()

        self.encoder = ViTEncoder()

        self.attention_pool = IJEPAAttentionPool(
            embed_dim=CFG.EMBED_DIM
        )

        self.classifier = nn.Sequential(
            nn.LayerNorm(CFG.EMBED_DIM),

            nn.Linear(
                CFG.EMBED_DIM,
                CFG.CLASSIFIER_HIDDEN_DIM
            ),

            nn.GELU(),
            nn.Dropout(
                CFG.CLASSIFIER_DROPOUT
            ),

            nn.Linear(
                CFG.CLASSIFIER_HIDDEN_DIM,
                num_classes
            )
        )

    def forward(
        self,
        images: torch.Tensor,
        return_attention: bool = False
    ):

        patch_tokens = self.encoder(images)

        pooled_features, attention_weights = (
            self.attention_pool(
                patch_tokens
            )
        )

        logits = self.classifier(
            pooled_features
        )

        if return_attention:
            return logits, attention_weights

        return logits


# ============================================================
# 7. IMAGE TRANSFORMS
# ============================================================

def get_train_transform():

    return transforms.Compose([
        transforms.Resize(
            (CFG.IMG_SIZE, CFG.IMG_SIZE),
            interpolation=(
                transforms.InterpolationMode.BICUBIC
            )
        ),

        transforms.RandomHorizontalFlip(
            p=0.5
        ),

        transforms.RandomApply([
            transforms.RandomAffine(
                degrees=7,
                translate=(0.04, 0.04),
                scale=(0.95, 1.05),
                shear=2
            )
        ], p=0.5),

        transforms.RandomApply([
            transforms.ColorJitter(
                brightness=0.10,
                contrast=0.12,
                saturation=0.03,
                hue=0.0
            )
        ], p=0.30),

        transforms.ToTensor(),

        transforms.Normalize(
            mean=[0.485, 0.456, 0.406],
            std=[0.229, 0.224, 0.225]
        )
    ])


def get_eval_transform():

    return transforms.Compose([
        transforms.Resize(
            (CFG.IMG_SIZE, CFG.IMG_SIZE),
            interpolation=(
                transforms.InterpolationMode.BICUBIC
            )
        ),

        transforms.ToTensor(),

        transforms.Normalize(
            mean=[0.485, 0.456, 0.406],
            std=[0.229, 0.224, 0.225]
        )
    ])


# ============================================================
# 8. DATASET WRAPPER FOR DIFFERENT TRANSFORMS
# ============================================================

class TransformSubset(Dataset):

    def __init__(
        self,
        base_dataset,
        indices: List[int],
        transform
    ):
        self.base_dataset = base_dataset
        self.indices = list(indices)
        self.transform = transform

        self.classes = base_dataset.classes
        self.class_to_idx = base_dataset.class_to_idx

        self.samples = [
            base_dataset.samples[index]
            for index in self.indices
        ]

        self.targets = [
            base_dataset.targets[index]
            for index in self.indices
        ]

    def __len__(self):
        return len(self.indices)

    def __getitem__(self, position):

        original_index = self.indices[position]

        image_path, label = (
            self.base_dataset.samples[
                original_index
            ]
        )

        with Image.open(image_path) as image:
            image = image.convert("RGB")

        image = self.transform(image)

        return image, label, image_path


class ImageFolderWithPath(datasets.ImageFolder):

    def __getitem__(self, index):

        image, label = super().__getitem__(
            index
        )

        image_path = self.samples[index][0]

        return image, label, image_path


# ============================================================
# 9. DATA SPLITTING
# ============================================================

def validate_split_ratios():

    total = (
        CFG.TRAIN_RATIO +
        CFG.VAL_RATIO +
        CFG.TEST_RATIO
    )

    if not np.isclose(total, 1.0):
        raise ValueError(
            "TRAIN_RATIO + VAL_RATIO + TEST_RATIO "
            "must equal 1.0."
        )


def build_auto_split_datasets():

    validate_split_ratios()

    base_dataset = datasets.ImageFolder(
        CFG.DATA_DIR,
        transform=None
    )

    targets = np.asarray(
        base_dataset.targets
    )

    all_indices = np.arange(
        len(base_dataset)
    )

    train_indices, temporary_indices = (
        train_test_split(
            all_indices,
            test_size=(
                CFG.VAL_RATIO +
                CFG.TEST_RATIO
            ),
            stratify=targets,
            random_state=CFG.SEED
        )
    )

    temporary_targets = targets[
        temporary_indices
    ]

    relative_test_ratio = (
        CFG.TEST_RATIO /
        (
            CFG.VAL_RATIO +
            CFG.TEST_RATIO
        )
    )

    val_indices, test_indices = train_test_split(
        temporary_indices,
        test_size=relative_test_ratio,
        stratify=temporary_targets,
        random_state=CFG.SEED
    )

    train_dataset = TransformSubset(
        base_dataset=base_dataset,
        indices=train_indices,
        transform=get_train_transform()
    )

    val_dataset = TransformSubset(
        base_dataset=base_dataset,
        indices=val_indices,
        transform=get_eval_transform()
    )

    test_dataset = TransformSubset(
        base_dataset=base_dataset,
        indices=test_indices,
        transform=get_eval_transform()
    )

    return (
        train_dataset,
        val_dataset,
        test_dataset,
        base_dataset.classes
    )


def build_presplit_datasets():

    train_path = os.path.join(
        CFG.DATA_DIR,
        "train"
    )

    val_path = os.path.join(
        CFG.DATA_DIR,
        "val"
    )

    test_path = os.path.join(
        CFG.DATA_DIR,
        "test"
    )

    for split_path in [
        train_path,
        val_path,
        test_path
    ]:
        if not os.path.isdir(split_path):
            raise FileNotFoundError(
                f"Dataset split not found:\n"
                f"{split_path}"
            )

    train_dataset = ImageFolderWithPath(
        train_path,
        transform=get_train_transform()
    )

    val_dataset = ImageFolderWithPath(
        val_path,
        transform=get_eval_transform()
    )

    test_dataset = ImageFolderWithPath(
        test_path,
        transform=get_eval_transform()
    )

    if not (
        train_dataset.classes ==
        val_dataset.classes ==
        test_dataset.classes
    ):
        raise ValueError(
            "Train, validation, and test class names "
            "do not match."
        )

    return (
        train_dataset,
        val_dataset,
        test_dataset,
        train_dataset.classes
    )


# ============================================================
# 10. CLASS DISTRIBUTION AND WEIGHTS
# ============================================================

def get_dataset_targets(
    dataset
) -> np.ndarray:

    if hasattr(dataset, "targets"):
        return np.asarray(
            dataset.targets
        )

    raise AttributeError(
        "Dataset does not expose targets."
    )


def calculate_class_weights(
    targets: np.ndarray,
    num_classes: int
) -> torch.Tensor:

    counts = np.bincount(
        targets,
        minlength=num_classes
    )

    weights = (
        len(targets) /
        (
            num_classes *
            np.maximum(counts, 1)
        )
    )

    print("Training class counts:", counts)
    print("Class weights:", weights)

    return torch.tensor(
        weights,
        dtype=torch.float32,
        device=DEVICE
    )


def create_weighted_sampler(
    targets: np.ndarray,
    num_classes: int
):

    counts = np.bincount(
        targets,
        minlength=num_classes
    )

    class_weights = 1.0 / np.maximum(
        counts,
        1
    )

    sample_weights = class_weights[
        targets
    ]

    return WeightedRandomSampler(
        weights=torch.tensor(
            sample_weights,
            dtype=torch.double
        ),
        num_samples=len(sample_weights),
        replacement=True
    )


# ============================================================
# 11. DATA LOADERS
# ============================================================

def build_dataloaders():

    if CFG.AUTO_SPLIT:
        (
            train_dataset,
            val_dataset,
            test_dataset,
            class_names
        ) = build_auto_split_datasets()

    else:
        (
            train_dataset,
            val_dataset,
            test_dataset,
            class_names
        ) = build_presplit_datasets()

    train_targets = get_dataset_targets(
        train_dataset
    )

    sampler = None
    shuffle = True

    if CFG.USE_WEIGHTED_SAMPLER:
        sampler = create_weighted_sampler(
            targets=train_targets,
            num_classes=len(class_names)
        )

        shuffle = False

    loader_options = {
        "batch_size": CFG.BATCH_SIZE,
        "num_workers": CFG.NUM_WORKERS,
        "pin_memory": CFG.PIN_MEMORY
    }

    if CFG.NUM_WORKERS > 0:
        loader_options["persistent_workers"] = (
            CFG.PERSISTENT_WORKERS
        )

        loader_options["prefetch_factor"] = (
            CFG.PREFETCH_FACTOR
        )

    train_loader = DataLoader(
        train_dataset,
        shuffle=shuffle,
        sampler=sampler,
        drop_last=False,
        **loader_options
    )

    val_loader = DataLoader(
        val_dataset,
        shuffle=False,
        drop_last=False,
        **loader_options
    )

    test_loader = DataLoader(
        test_dataset,
        shuffle=False,
        drop_last=False,
        **loader_options
    )

    print("\nDataset summary")
    print("-------------------------------")
    print("Classes          :", class_names)
    print("Training images  :", len(train_dataset))
    print("Validation images:", len(val_dataset))
    print("Test images      :", len(test_dataset))

    return (
        train_loader,
        val_loader,
        test_loader,
        class_names,
        train_targets,
        test_dataset
    )


# ============================================================
# 12. FREEZING AND UNFREEZING
# ============================================================

def freeze_encoder(
    model: IJEPAClassifier
) -> None:

    for parameter in model.encoder.parameters():
        parameter.requires_grad = False


def unfreeze_last_encoder_blocks(
    model: IJEPAClassifier,
    number_of_blocks: int
) -> None:

    freeze_encoder(model)

    for block in model.encoder.blocks[
        -number_of_blocks:
    ]:
        for parameter in block.parameters():
            parameter.requires_grad = True

    for parameter in model.encoder.norm.parameters():
        parameter.requires_grad = True


def unfreeze_complete_encoder(
    model: IJEPAClassifier
) -> None:

    for parameter in model.encoder.parameters():
        parameter.requires_grad = True


def count_trainable_parameters(
    model: nn.Module
) -> int:

    return sum(
        parameter.numel()
        for parameter in model.parameters()
        if parameter.requires_grad
    )


# ============================================================
# 13. OPTIMIZERS
# ============================================================

def build_optimizer(
    model: IJEPAClassifier,
    phase_name: str
):

    if phase_name == "head":

        freeze_encoder(model)

        parameter_groups = [
            {
                "params": (
                    list(
                        model.attention_pool.parameters()
                    ) +
                    list(
                        model.classifier.parameters()
                    )
                ),
                "lr": CFG.HEAD_LR
            }
        ]

    elif phase_name == "partial":

        unfreeze_last_encoder_blocks(
            model,
            CFG.NUM_UNFROZEN_BLOCKS
        )

        parameter_groups = [
            {
                "params": [
                    parameter
                    for parameter
                    in model.encoder.parameters()
                    if parameter.requires_grad
                ],
                "lr": CFG.PARTIAL_ENCODER_LR
            },
            {
                "params": (
                    list(
                        model.attention_pool.parameters()
                    ) +
                    list(
                        model.classifier.parameters()
                    )
                ),
                "lr": CFG.PARTIAL_HEAD_LR
            }
        ]

    elif phase_name == "full":

        unfreeze_complete_encoder(
            model
        )

        parameter_groups = [
            {
                "params": (
                    model.encoder.parameters()
                ),
                "lr": CFG.FULL_ENCODER_LR
            },
            {
                "params": (
                    list(
                        model.attention_pool.parameters()
                    ) +
                    list(
                        model.classifier.parameters()
                    )
                ),
                "lr": CFG.FULL_HEAD_LR
            }
        ]

    else:
        raise ValueError(
            f"Unknown training phase: {phase_name}"
        )

    optimizer = torch.optim.AdamW(
        parameter_groups,
        weight_decay=CFG.WEIGHT_DECAY,
        betas=(0.9, 0.999)
    )

    print(
        f"\nPhase: {phase_name}"
    )

    print(
        "Trainable parameters:",
        f"{count_trainable_parameters(model):,}"
    )

    return optimizer


# ============================================================
# 14. TRAINING AND EVALUATION
# ============================================================

def calculate_basic_metrics(
    labels: np.ndarray,
    predictions: np.ndarray
) -> Dict[str, float]:

    return {
        "accuracy": accuracy_score(
            labels,
            predictions
        ),

        "balanced_accuracy": (
            balanced_accuracy_score(
                labels,
                predictions
            )
        ),

        "precision_macro": precision_score(
            labels,
            predictions,
            average="macro",
            zero_division=0
        ),

        "recall_macro": recall_score(
            labels,
            predictions,
            average="macro",
            zero_division=0
        ),

        "f1_macro": f1_score(
            labels,
            predictions,
            average="macro",
            zero_division=0
        ),

        "f1_weighted": f1_score(
            labels,
            predictions,
            average="weighted",
            zero_division=0
        )
    }


def train_one_epoch(
    model,
    loader,
    criterion,
    optimizer,
    scaler,
    amp_enabled
):

    model.train()

    total_loss = 0.0

    all_labels = []
    all_predictions = []

    progress_bar = tqdm(
        loader,
        desc="Training",
        leave=False
    )

    for images, labels, _ in progress_bar:

        images = images.to(
            DEVICE,
            non_blocking=True
        )

        labels = labels.to(
            DEVICE,
            non_blocking=True
        )

        optimizer.zero_grad(
            set_to_none=True
        )

        with torch.autocast(
            device_type=DEVICE.type,
            dtype=torch.float16,
            enabled=amp_enabled
        ):
            logits = model(images)

            loss = criterion(
                logits,
                labels
            )

        if not torch.isfinite(loss):
            print(
                "Skipping non-finite training loss."
            )
            continue

        scaler.scale(loss).backward()

        scaler.unscale_(
            optimizer
        )

        torch.nn.utils.clip_grad_norm_(
            [
                parameter
                for parameter
                in model.parameters()
                if parameter.requires_grad
            ],
            max_norm=CFG.GRADIENT_CLIP_NORM
        )

        scaler.step(
            optimizer
        )

        scaler.update()

        predictions = torch.argmax(
            logits,
            dim=1
        )

        total_loss += (
            loss.item() *
            images.size(0)
        )

        all_labels.extend(
            labels.detach().cpu().numpy()
        )

        all_predictions.extend(
            predictions.detach().cpu().numpy()
        )

        progress_bar.set_postfix({
            "loss": f"{loss.item():.4f}"
        })

    average_loss = (
        total_loss /
        len(loader.dataset)
    )

    metrics = calculate_basic_metrics(
        np.asarray(all_labels),
        np.asarray(all_predictions)
    )

    return average_loss, metrics


@torch.no_grad()
def evaluate_model(
    model,
    loader,
    criterion
):

    model.eval()

    total_loss = 0.0

    all_labels = []
    all_predictions = []
    all_probabilities = []
    all_paths = []

    for images, labels, paths in tqdm(
        loader,
        desc="Evaluating",
        leave=False
    ):

        images = images.to(
            DEVICE,
            non_blocking=True
        )

        labels = labels.to(
            DEVICE,
            non_blocking=True
        )

        logits = model(images)

        loss = criterion(
            logits,
            labels
        )

        probabilities = torch.softmax(
            logits,
            dim=1
        )

        predictions = torch.argmax(
            probabilities,
            dim=1
        )

        total_loss += (
            loss.item() *
            images.size(0)
        )

        all_labels.extend(
            labels.cpu().numpy()
        )

        all_predictions.extend(
            predictions.cpu().numpy()
        )

        all_probabilities.extend(
            probabilities.cpu().numpy()
        )

        all_paths.extend(
            list(paths)
        )

    labels = np.asarray(
        all_labels
    )

    predictions = np.asarray(
        all_predictions
    )

    probabilities = np.asarray(
        all_probabilities
    )

    average_loss = (
        total_loss /
        len(loader.dataset)
    )

    metrics = calculate_basic_metrics(
        labels,
        predictions
    )

    return {
        "loss": average_loss,
        "metrics": metrics,
        "labels": labels,
        "predictions": predictions,
        "probabilities": probabilities,
        "paths": all_paths
    }


# ============================================================
# 15. COMPLETE STAGED TRAINING
# ============================================================

def train_classification_model(
    model,
    train_loader,
    val_loader,
    criterion
):

    phases = [
        {
            "name": "head",
            "epochs": CFG.HEAD_WARMUP_EPOCHS
        },
        {
            "name": "partial",
            "epochs": CFG.PARTIAL_FINETUNE_EPOCHS
        },
        {
            "name": "full",
            "epochs": CFG.FULL_FINETUNE_EPOCHS
        }
    ]

    amp_enabled = (
        CFG.USE_AMP and
        DEVICE.type == "cuda"
    )

    scaler = torch.cuda.amp.GradScaler(
        enabled=amp_enabled
    )

    history = []

    best_state = None
    best_val_f1 = -np.inf
    patience_counter = 0
    global_epoch = 0

    for phase in phases:

        if phase["epochs"] <= 0:
            continue

        optimizer = build_optimizer(
            model,
            phase["name"]
        )

        scheduler = (
            torch.optim.lr_scheduler.CosineAnnealingLR(
                optimizer,
                T_max=max(
                    1,
                    phase["epochs"]
                ),
                eta_min=1e-7
            )
        )

        for phase_epoch in range(
            phase["epochs"]
        ):

            global_epoch += 1

            print(
                f"\nEpoch {global_epoch} | "
                f"Phase: {phase['name']} | "
                f"Phase epoch: "
                f"{phase_epoch + 1}/"
                f"{phase['epochs']}"
            )

            train_loss, train_metrics = (
                train_one_epoch(
                    model=model,
                    loader=train_loader,
                    criterion=criterion,
                    optimizer=optimizer,
                    scaler=scaler,
                    amp_enabled=amp_enabled
                )
            )

            val_result = evaluate_model(
                model=model,
                loader=val_loader,
                criterion=criterion
            )

            val_loss = val_result["loss"]
            val_metrics = val_result["metrics"]

            current_lr = optimizer.param_groups[
                0
            ]["lr"]

            record = {
                "epoch": global_epoch,
                "phase": phase["name"],
                "learning_rate": current_lr,

                "train_loss": train_loss,
                "val_loss": val_loss,

                "train_accuracy": (
                    train_metrics["accuracy"]
                ),

                "val_accuracy": (
                    val_metrics["accuracy"]
                ),

                "train_balanced_accuracy": (
                    train_metrics[
                        "balanced_accuracy"
                    ]
                ),

                "val_balanced_accuracy": (
                    val_metrics[
                        "balanced_accuracy"
                    ]
                ),

                "train_f1_macro": (
                    train_metrics["f1_macro"]
                ),

                "val_f1_macro": (
                    val_metrics["f1_macro"]
                ),

                "train_f1_weighted": (
                    train_metrics["f1_weighted"]
                ),

                "val_f1_weighted": (
                    val_metrics["f1_weighted"]
                )
            }

            history.append(record)

            print(
                f"Train Loss: {train_loss:.4f} | "
                f"Val Loss: {val_loss:.4f}"
            )

            print(
                f"Train Acc: "
                f"{train_metrics['accuracy']:.4f} | "
                f"Val Acc: "
                f"{val_metrics['accuracy']:.4f}"
            )

            print(
                f"Train Macro-F1: "
                f"{train_metrics['f1_macro']:.4f} | "
                f"Val Macro-F1: "
                f"{val_metrics['f1_macro']:.4f}"
            )

            if (
                val_metrics["f1_macro"] >
                best_val_f1
            ):
                best_val_f1 = (
                    val_metrics["f1_macro"]
                )

                best_state = deepcopy(
                    model.state_dict()
                )

                patience_counter = 0

                torch.save(
                    {
                        "model_state_dict": best_state,
                        "class_names": None,
                        "epoch": global_epoch,
                        "val_f1_macro": best_val_f1,
                        "config": asdict(CFG)
                    },
                    os.path.join(
                        CFG.SAVE_DIR,
                        "best_classifier.pth"
                    )
                )

                print(
                    "Saved new best classifier."
                )

            else:
                patience_counter += 1

            scheduler.step()

            pd.DataFrame(
                history
            ).to_csv(
                os.path.join(
                    CFG.SAVE_DIR,
                    "training_history.csv"
                ),
                index=False
            )

            if (
                patience_counter >=
                CFG.EARLY_STOPPING_PATIENCE
            ):
                print(
                    "Early stopping activated."
                )

                if best_state is not None:
                    model.load_state_dict(
                        best_state
                    )

                return model, history

    if best_state is not None:
        model.load_state_dict(
            best_state
        )

    return model, history


# ============================================================
# 16. PLOT TRAINING CURVES
# ============================================================

def plot_training_curves(
    history: List[dict]
):

    dataframe = pd.DataFrame(
        history
    )

    epochs = dataframe["epoch"]

    plt.figure(figsize=(9, 6))

    plt.plot(
        epochs,
        dataframe["train_loss"],
        label="Training Loss",
        linewidth=2
    )

    plt.plot(
        epochs,
        dataframe["val_loss"],
        label="Validation Loss",
        linewidth=2
    )

    plt.xlabel("Epoch")
    plt.ylabel("Loss")
    plt.title(
        "Training and Validation Loss"
    )
    plt.legend()
    plt.grid(True, alpha=0.3)
    plt.tight_layout()

    plt.savefig(
        os.path.join(
            CFG.SAVE_DIR,
            "loss_curve.png"
        ),
        dpi=300
    )

    plt.close()

    plt.figure(figsize=(9, 6))

    plt.plot(
        epochs,
        dataframe["train_accuracy"],
        label="Training Accuracy",
        linewidth=2
    )

    plt.plot(
        epochs,
        dataframe["val_accuracy"],
        label="Validation Accuracy",
        linewidth=2
    )

    plt.xlabel("Epoch")
    plt.ylabel("Accuracy")
    plt.title(
        "Training and Validation Accuracy"
    )
    plt.legend()
    plt.grid(True, alpha=0.3)
    plt.tight_layout()

    plt.savefig(
        os.path.join(
            CFG.SAVE_DIR,
            "accuracy_curve.png"
        ),
        dpi=300
    )

    plt.close()

    plt.figure(figsize=(9, 6))

    plt.plot(
        epochs,
        dataframe["train_f1_macro"],
        label="Training Macro-F1",
        linewidth=2
    )

    plt.plot(
        epochs,
        dataframe["val_f1_macro"],
        label="Validation Macro-F1",
        linewidth=2
    )

    plt.xlabel("Epoch")
    plt.ylabel("Macro-F1")
    plt.title(
        "Training and Validation Macro-F1"
    )
    plt.legend()
    plt.grid(True, alpha=0.3)
    plt.tight_layout()

    plt.savefig(
        os.path.join(
            CFG.SAVE_DIR,
            "macro_f1_curve.png"
        ),
        dpi=300
    )

    plt.close()


# ============================================================
# 17. CONFUSION MATRIX
# ============================================================

def plot_confusion_matrix_manual(
    matrix: np.ndarray,
    class_names: List[str],
    filename: str,
    title: str,
    normalized: bool = False
):

    plt.figure(
        figsize=(
            max(8, len(class_names) * 1.25),
            max(7, len(class_names))
        )
    )

    plt.imshow(
        matrix,
        interpolation="nearest",
        cmap="Blues"
    )

    plt.title(title)
    plt.colorbar()

    tick_marks = np.arange(
        len(class_names)
    )

    plt.xticks(
        tick_marks,
        class_names,
        rotation=45,
        ha="right"
    )

    plt.yticks(
        tick_marks,
        class_names
    )

    threshold = (
        matrix.max() / 2.0
        if matrix.size > 0
        else 0
    )

    for row in range(
        matrix.shape[0]
    ):
        for column in range(
            matrix.shape[1]
        ):

            if normalized:
                text = (
                    f"{matrix[row, column]:.2f}"
                )
            else:
                text = (
                    f"{int(matrix[row, column])}"
                )

            plt.text(
                column,
                row,
                text,
                horizontalalignment="center",
                verticalalignment="center",
                color=(
                    "white"
                    if matrix[row, column] >
                    threshold
                    else "black"
                )
            )

    plt.ylabel("True Label")
    plt.xlabel("Predicted Label")
    plt.tight_layout()

    plt.savefig(
        os.path.join(
            CFG.SAVE_DIR,
            filename
        ),
        dpi=300
    )

    plt.close()


# ============================================================
# 18. CLASS-WISE SENSITIVITY AND SPECIFICITY
# ============================================================

def calculate_classwise_statistics(
    labels,
    predictions,
    class_names
):

    matrix = confusion_matrix(
        labels,
        predictions,
        labels=np.arange(
            len(class_names)
        )
    )

    rows = []

    total = matrix.sum()

    for class_index, class_name in enumerate(
        class_names
    ):

        true_positive = matrix[
            class_index,
            class_index
        ]

        false_negative = (
            matrix[class_index, :].sum() -
            true_positive
        )

        false_positive = (
            matrix[:, class_index].sum() -
            true_positive
        )

        true_negative = (
            total -
            true_positive -
            false_negative -
            false_positive
        )

        sensitivity = (
            true_positive /
            max(
                true_positive +
                false_negative,
                1
            )
        )

        specificity = (
            true_negative /
            max(
                true_negative +
                false_positive,
                1
            )
        )

        precision = (
            true_positive /
            max(
                true_positive +
                false_positive,
                1
            )
        )

        rows.append({
            "class": class_name,
            "sensitivity": sensitivity,
            "specificity": specificity,
            "precision": precision,
            "support": int(
                matrix[class_index, :].sum()
            )
        })

    return pd.DataFrame(rows)


# ============================================================
# 19. ROC CURVES
# ============================================================

def plot_multiclass_roc(
    labels,
    probabilities,
    class_names
):

    num_classes = len(class_names)

    if num_classes == 2:

        fpr, tpr, _ = roc_curve(
            labels,
            probabilities[:, 1]
        )

        roc_auc = auc(
            fpr,
            tpr
        )

        plt.figure(figsize=(8, 7))

        plt.plot(
            fpr,
            tpr,
            linewidth=2,
            label=(
                f"{class_names[1]} "
                f"(AUC={roc_auc:.4f})"
            )
        )

    else:

        binary_labels = label_binarize(
            labels,
            classes=np.arange(
                num_classes
            )
        )

        plt.figure(figsize=(10, 8))

        for class_index, class_name in enumerate(
            class_names
        ):

            if len(
                np.unique(
                    binary_labels[:, class_index]
                )
            ) < 2:
                continue

            fpr, tpr, _ = roc_curve(
                binary_labels[:, class_index],
                probabilities[:, class_index]
            )

            roc_auc = auc(
                fpr,
                tpr
            )

            plt.plot(
                fpr,
                tpr,
                linewidth=2,
                label=(
                    f"{class_name} "
                    f"(AUC={roc_auc:.4f})"
                )
            )

    plt.plot(
        [0, 1],
        [0, 1],
        linestyle="--",
        linewidth=1.5
    )

    plt.xlabel(
        "False Positive Rate"
    )

    plt.ylabel(
        "True Positive Rate"
    )

    plt.title(
        "Receiver Operating Characteristic Curves"
    )

    plt.legend(
        loc="lower right",
        fontsize=8
    )

    plt.grid(
        True,
        alpha=0.3
    )

    plt.tight_layout()

    plt.savefig(
        os.path.join(
            CFG.SAVE_DIR,
            "roc_curves.png"
        ),
        dpi=300
    )

    plt.close()


# ============================================================
# 20. PRECISION-RECALL CURVES
# ============================================================

def plot_precision_recall_curves(
    labels,
    probabilities,
    class_names
):

    num_classes = len(class_names)

    if num_classes == 2:

        precision, recall, _ = (
            precision_recall_curve(
                labels,
                probabilities[:, 1]
            )
        )

        average_precision = (
            average_precision_score(
                labels,
                probabilities[:, 1]
            )
        )

        plt.figure(figsize=(8, 7))

        plt.plot(
            recall,
            precision,
            linewidth=2,
            label=(
                f"{class_names[1]} "
                f"(AP={average_precision:.4f})"
            )
        )

    else:

        binary_labels = label_binarize(
            labels,
            classes=np.arange(
                num_classes
            )
        )

        plt.figure(figsize=(10, 8))

        for class_index, class_name in enumerate(
            class_names
        ):

            precision, recall, _ = (
                precision_recall_curve(
                    binary_labels[:, class_index],
                    probabilities[:, class_index]
                )
            )

            average_precision = (
                average_precision_score(
                    binary_labels[:, class_index],
                    probabilities[:, class_index]
                )
            )

            plt.plot(
                recall,
                precision,
                linewidth=2,
                label=(
                    f"{class_name} "
                    f"(AP={average_precision:.4f})"
                )
            )

    plt.xlabel("Recall")
    plt.ylabel("Precision")

    plt.title(
        "Precision-Recall Curves"
    )

    plt.legend(
        loc="best",
        fontsize=8
    )

    plt.grid(
        True,
        alpha=0.3
    )

    plt.tight_layout()

    plt.savefig(
        os.path.join(
            CFG.SAVE_DIR,
            "precision_recall_curves.png"
        ),
        dpi=300
    )

    plt.close()


# ============================================================
# 21. SAVE TEST PREDICTIONS WITH CONFIDENCE
# ============================================================

def save_test_predictions(
    result,
    class_names
):

    rows = []

    for index in range(
        len(result["labels"])
    ):

        true_index = int(
            result["labels"][index]
        )

        predicted_index = int(
            result["predictions"][index]
        )

        confidence = float(
            result["probabilities"][
                index,
                predicted_index
            ]
        )

        row = {
            "image_path": (
                result["paths"][index]
            ),

            "true_label": (
                class_names[true_index]
            ),

            "predicted_label": (
                class_names[predicted_index]
            ),

            "confidence": confidence,

            "correct": (
                true_index ==
                predicted_index
            )
        }

        for class_index, class_name in enumerate(
            class_names
        ):
            row[
                f"probability_{class_name}"
            ] = float(
                result["probabilities"][
                    index,
                    class_index
                ]
            )

        rows.append(row)

    dataframe = pd.DataFrame(
        rows
    )

    dataframe.to_csv(
        os.path.join(
            CFG.SAVE_DIR,
            "test_predictions_with_confidence.csv"
        ),
        index=False
    )

    return dataframe


# ============================================================
# 22. QUALITATIVE TEST PREDICTIONS
# ============================================================

def show_test_predictions(
    prediction_dataframe,
    number_of_samples=12
):

    if len(prediction_dataframe) == 0:
        return

    sampled_dataframe = (
        prediction_dataframe.sample(
            n=min(
                number_of_samples,
                len(prediction_dataframe)
            ),
            random_state=CFG.SEED
        )
        .reset_index(drop=True)
    )

    columns = 3
    rows = math.ceil(
        len(sampled_dataframe) /
        columns
    )

    figure = plt.figure(
        figsize=(15, rows * 5)
    )

    for index, row in sampled_dataframe.iterrows():

        axis = figure.add_subplot(
            rows,
            columns,
            index + 1
        )

        with Image.open(
            row["image_path"]
        ) as image:
            image = image.convert("RGB")

        axis.imshow(image)

        correctness = (
            "Correct"
            if row["correct"]
            else "Incorrect"
        )

        title = (
            f"True: {row['true_label']}\n"
            f"Predicted: "
            f"{row['predicted_label']}\n"
            f"Confidence: "
            f"{row['confidence']:.2%}\n"
            f"{correctness}"
        )

        axis.set_title(
            title,
            fontsize=10
        )

        axis.axis("off")

    plt.tight_layout()

    plt.savefig(
        os.path.join(
            CFG.SAVE_DIR,
            "qualitative_test_predictions.png"
        ),
        dpi=300,
        bbox_inches="tight"
    )

    plt.close()


# ============================================================
# 23. FINAL TEST EVALUATION
# ============================================================

def perform_final_evaluation(
    model,
    test_loader,
    criterion,
    class_names
):

    print(
        "\nRunning final test evaluation..."
    )

    result = evaluate_model(
        model=model,
        loader=test_loader,
        criterion=criterion
    )

    labels = result["labels"]
    predictions = result["predictions"]
    probabilities = result[
        "probabilities"
    ]

    print(
        f"\nTest Loss: "
        f"{result['loss']:.6f}"
    )

    for metric_name, metric_value in (
        result["metrics"].items()
    ):
        print(
            f"{metric_name}: "
            f"{metric_value:.6f}"
        )

    report_text = classification_report(
        labels,
        predictions,
        target_names=class_names,
        digits=4,
        zero_division=0
    )

    print(
        "\nClassification Report\n"
    )

    print(report_text)

    with open(
        os.path.join(
            CFG.SAVE_DIR,
            "classification_report.txt"
        ),
        "w"
    ) as file:
        file.write(report_text)

    report_dictionary = (
        classification_report(
            labels,
            predictions,
            target_names=class_names,
            output_dict=True,
            zero_division=0
        )
    )

    pd.DataFrame(
        report_dictionary
    ).transpose().to_csv(
        os.path.join(
            CFG.SAVE_DIR,
            "classification_report.csv"
        )
    )

    matrix = confusion_matrix(
        labels,
        predictions,
        labels=np.arange(
            len(class_names)
        )
    )

    plot_confusion_matrix_manual(
        matrix=matrix,
        class_names=class_names,
        filename="confusion_matrix.png",
        title="Confusion Matrix",
        normalized=False
    )

    normalized_matrix = (
        matrix.astype(np.float64) /
        np.maximum(
            matrix.sum(
                axis=1,
                keepdims=True
            ),
            1
        )
    )

    plot_confusion_matrix_manual(
        matrix=normalized_matrix,
        class_names=class_names,
        filename=(
            "normalized_confusion_matrix.png"
        ),
        title="Normalized Confusion Matrix",
        normalized=True
    )

    classwise_statistics = (
        calculate_classwise_statistics(
            labels=labels,
            predictions=predictions,
            class_names=class_names
        )
    )

    classwise_statistics.to_csv(
        os.path.join(
            CFG.SAVE_DIR,
            "classwise_sensitivity_specificity.csv"
        ),
        index=False
    )

    plot_multiclass_roc(
        labels=labels,
        probabilities=probabilities,
        class_names=class_names
    )

    plot_precision_recall_curves(
        labels=labels,
        probabilities=probabilities,
        class_names=class_names
    )

    num_classes = len(class_names)

    try:
        if num_classes == 2:
            roc_auc = roc_auc_score(
                labels,
                probabilities[:, 1]
            )

        else:
            roc_auc = roc_auc_score(
                labels,
                probabilities,
                multi_class="ovr",
                average="macro"
            )

    except ValueError:
        roc_auc = np.nan

    final_metrics = {
        "test_loss": result["loss"],
        **result["metrics"],
        "roc_auc": roc_auc
    }

    pd.DataFrame(
        [final_metrics]
    ).to_csv(
        os.path.join(
            CFG.SAVE_DIR,
            "final_test_metrics.csv"
        ),
        index=False
    )

    prediction_dataframe = (
        save_test_predictions(
            result=result,
            class_names=class_names
        )
    )

    show_test_predictions(
        prediction_dataframe,
        number_of_samples=12
    )

    return result


# ============================================================
# 24. MAIN
# ============================================================

def main():

    print("=" * 70)
    print(
        "STAGE 2A: I-JEPA CLASSIFICATION FINE-TUNING"
    )
    print("=" * 70)

    print("Device:", DEVICE)

    if torch.cuda.is_available():
        print(
            "GPU:",
            torch.cuda.get_device_name(0)
        )

    (
        train_loader,
        val_loader,
        test_loader,
        class_names,
        train_targets,
        test_dataset
    ) = build_dataloaders()

    with open(
        os.path.join(
            CFG.SAVE_DIR,
            "class_names.json"
        ),
        "w"
    ) as file:
        json.dump(
            class_names,
            file,
            indent=4
        )

    model = IJEPAClassifier(
        num_classes=len(class_names)
    )

    model.encoder = (
        load_pretrained_ijepa_encoder(
            encoder=model.encoder,
            checkpoint_path=(
                CFG.PRETRAINED_ENCODER_PATH
            ),
            strict=True
        )
    )

    model = model.to(
        DEVICE
    )

    class_weights = None

    if (
        CFG.USE_CLASS_WEIGHTS and
        not CFG.USE_WEIGHTED_SAMPLER
    ):
        class_weights = calculate_class_weights(
            targets=train_targets,
            num_classes=len(class_names)
        )

    criterion = nn.CrossEntropyLoss(
        weight=class_weights,
        label_smoothing=(
            CFG.LABEL_SMOOTHING
        )
    )

    model, history = (
        train_classification_model(
            model=model,
            train_loader=train_loader,
            val_loader=val_loader,
            criterion=criterion
        )
    )

    plot_training_curves(
        history
    )

    # Save final best model with class names.
    torch.save(
        {
            "model_state_dict": (
                model.state_dict()
            ),

            "class_names": class_names,

            "config": asdict(CFG)
        },
        os.path.join(
            CFG.SAVE_DIR,
            "final_best_classifier.pth"
        )
    )

    perform_final_evaluation(
        model=model,
        test_loader=test_loader,
        criterion=criterion,
        class_names=class_names
    )

    print(
        "\nStage 2A completed successfully."
    )

    print(
        "All results are stored in:\n",
        os.path.abspath(
            CFG.SAVE_DIR
        )
    )


# ============================================================
# 25. EXECUTION
# ============================================================

if __name__ == "__main__":
    main()

STAGE 2A: I-JEPA CLASSIFICATION FINE-TUNING
Device: cuda
GPU: NVIDIA RTX PRO 4000 Blackwell

Dataset summary
-------------------------------
Classes          : ['Fetal abdomen', 'Fetal brain', 'Fetal femur', 'Fetal thorax', 'Maternal cervix', 'Other']
Training images  : 3596
Validation images: 771
Test images      : 771
Loading pretrained encoder:
/home/user/workspace/amar/SSL_model/checkpoints_ultrasound_ijepa/final_ijepa_encoder.pth
Pretrained I-JEPA encoder loaded.
Training class counts: [ 234  900  326  502  455 1179]
Class weights: [2.56125356 0.66592593 1.83844581 1.1938911  1.31721612 0.5083404 ]

Phase: head
Trainable parameters: 174,983

Epoch 1 | Phase: head | Phase epoch: 1/5


Training:   0%|          | 0/113 [00:00<?, ?it/s]

Evaluating:   0%|          | 0/25 [00:00<?, ?it/s]

Train Loss: 0.9049 | Val Loss: 0.7309
Train Acc: 0.7300 | Val Acc: 0.8236
Train Macro-F1: 0.7006 | Val Macro-F1: 0.8086
Saved new best classifier.

Epoch 2 | Phase: head | Phase epoch: 2/5


Training:   0%|          | 0/113 [00:00<?, ?it/s]

...

Training:   0%|          | 0/113 [00:00<?, ?it/s]

Evaluating:   0%|          | 0/25 [00:00<?, ?it/s]

Train Loss: 0.3914 | Val Loss: 0.5239
Train Acc: 0.9627 | Val Acc: 0.9118
Train Macro-F1: 0.9604 | Val Macro-F1: 0.8983
Early stopping activated.

Running final test evaluation...


Evaluating:   0%|          | 0/25 [00:00<?, ?it/s]


Test Loss: 0.523429
accuracy: 0.922179
balanced_accuracy: 0.918495
precision_macro: 0.890342
recall_macro: 0.918495
f1_macro: 0.902907
f1_weighted: 0.923171

Classification Report

                 precision    recall  f1-score   support

  Fetal abdomen     0.7368    0.8400    0.7850        50
    Fetal brain     0.9845    0.9845    0.9845       193
    Fetal femur     0.7750    0.8857    0.8267        70
   Fetal thorax     0.9018    0.9352    0.9182       108
Maternal cervix     1.0000    1.0000    1.0000        97
          Other     0.9440    0.8656    0.9031       253

       accuracy                         0.9222       771
      macro avg     0.8903    0.9185    0.9029       771
   weighted avg     0.9265    0.9222    0.9232       771


Stage 2A completed successfully.
All results are stored in:
 /home/user/workspace/amar/SSL_model/stage2a_classification_results
